# Reproduce emergent planning in Sokoban

**Paper:** Emergent Planning in Model-Free Reinforcement Learning (ICLR 2025)  
**Default execution:** bounded synthetic smoke path, not a scientific reproduction  
**Current evidence:** label, probe, exact-occurrence, control, and intervention mechanics only; paper results are **not evaluated**.


## Method and scope

The label generator below follows the paper's future-dependent definitions. For each square and transition, **Agent Approach Direction** is the side (`UP`, `DOWN`, `LEFT`, or `RIGHT`) from which the agent next enters that square; **Box Push Direction** is the direction in which a box is next pushed off that square; `NEVER` means the event never occurs again in the episode. Labels require the completed trajectory and are never derived from probe predictions.

The scientific target is predeclared as 1x1 linear probes on cell states at `(layer=2, tick=2)`. Levels, not transitions or squares, define the held-out split. The report includes macro-F1 across all five classes, standard deviation over initialization seeds, and a level-bootstrap 95% interval whose statistic averages across those same seeds. The class-frequency control receives the same level bootstrap. Observation-only, shuffled-label, and class-frequency probes are controls. The temporal diagnostic applies the same final-occurrence probe to all nine exact `(layer, tick)` occurrences; it does not collapse repeated calls.

The intervention adds `w_RIGHT - w_NEVER` from the Box Push Direction probe at a predeclared grid square. The smoke policy declares its action order as `(UP, DOWN, LEFT, RIGHT)`; all action-direction comparisons use that mapping. Matched arms share the checkpoint, complete frozen evaluation tensors, random stream, and artifact identities, while each generated probe and vector has its own digest. Controls are the no-op arm, a shuffled-label probe vector, and a random vector matched to the trained vector's norm and signed projection onto the policy's action direction. Report decoded-plan changes separately from action changes. Matched inputs and controls establish the comparison mechanics, not causality.

The smoke path embeds future labels in a tiny recurrent fixture so every stage can be exercised quickly. It is not Sokoban, not a DRC checkpoint evaluation, not reference agreement, and not evidence that any model plans. Even a future paper-exact run would support only the analyzed checkpoint, task, level set, and predeclared intervention—not model-free RL broadly.


In [ ]:
import os

import torch
from tensordict import TensorDict
from tensordict.nn import TensorDictModule
from tdhook.latent import ActivationCaching, SteeringVectors
from tdhook.targets import Target
from tdhook.workflow import Workflow
from xdrl import Interaction, RecurrentSemantics, run_workflow

MODE = os.environ.get("XDRL_EMERGENT_PLANNING_MODE", "smoke")
SEED = 5519
DIRECTIONS = ("UP", "DOWN", "LEFT", "RIGHT")
ACTION_INDEX = {direction: index for index, direction in enumerate(DIRECTIONS)}
CLASSES = ("NEVER", *DIRECTIONS)
CLASS_FROM_DELTA = {(-1, 0): 1, (1, 0): 2, (0, -1): 3, (0, 1): 4}
SELECTED_COORDINATES = (("layer", 2), ("tick", 2))
PAPER_ASSET_RELEASE = None

if MODE not in {"smoke", "paper"}:
    raise ValueError("XDRL_EMERGENT_PLANNING_MODE must be 'smoke' or 'paper'")
if MODE == "paper" and PAPER_ASSET_RELEASE is None:
    raise RuntimeError(
        "paper-exact checkpoint, environment, labels, intervention levels, and reference code are unavailable; "
        "scientific mode is blocked rather than substituting the later learned-planner assets"
    )

torch.manual_seed(SEED)

## Derive future Agent Approach and Box Push labels

The generator accepts integer agent coordinates shaped `[level, time, 2]` and box coordinates shaped `[level, time, box, 2]`. A small hand-checked trajectory guards the directional asymmetry: entering a square **from the left** is `LEFT`, while a box moving right is `RIGHT`.


In [ ]:
def future_direction_labels(agent_xy, box_xy, height, width):
    if agent_xy.ndim != 3 or agent_xy.shape[-1] != 2:
        raise ValueError("agent_xy must have shape [level, time, 2]")
    if box_xy.ndim != 4 or box_xy.shape[:2] != agent_xy.shape[:2] or box_xy.shape[-1] != 2:
        raise ValueError("box_xy must have shape [level, time, box, 2]")
    levels, steps, _ = agent_xy.shape
    approach = torch.zeros(levels, steps, height, width, dtype=torch.long)
    push = torch.zeros_like(approach)
    for level in range(levels):
        for time in range(steps):
            for future in range(time + 1, steps):
                destination = agent_xy[level, future]
                source = agent_xy[level, future - 1]
                if not torch.equal(destination, source):
                    row, column = destination.tolist()
                    if approach[level, time, row, column] == 0:
                        side = tuple((source - destination).tolist())
                        approach[level, time, row, column] = CLASS_FROM_DELTA[side]
                for box in range(box_xy.shape[2]):
                    source = box_xy[level, future - 1, box]
                    destination = box_xy[level, future, box]
                    if not torch.equal(destination, source):
                        row, column = source.tolist()
                        if push[level, time, row, column] == 0:
                            direction = tuple((destination - source).tolist())
                            push[level, time, row, column] = CLASS_FROM_DELTA[direction]
    return approach, push


check_agent = torch.tensor([[[1, 0], [1, 1], [1, 2]]])
check_box = torch.tensor([[[[2, 0]], [[2, 1]], [[2, 2]]]])
check_approach, check_push = future_direction_labels(check_agent, check_box, 3, 3)
assert CLASSES[check_approach[0, 0, 1, 1]] == "LEFT"
assert CLASSES[check_push[0, 0, 2, 0]] == "RIGHT"

In [ ]:
def bounded_step(position, delta, height, width):
    candidate = position + torch.tensor(delta)
    if 0 <= candidate[0] < height and 0 <= candidate[1] < width:
        return candidate
    return position - torch.tensor(delta)


def smoke_trajectory_bundle(levels=30, steps=7, height=4, width=4):
    agent = torch.zeros(levels, steps, 2, dtype=torch.long)
    boxes = torch.zeros(levels, steps, 1, 2, dtype=torch.long)
    targets = torch.zeros(levels, 2, dtype=torch.long)
    agent_directions = ((0, 1), (1, 0), (0, -1), (-1, 0))
    box_directions = ((1, 0), (0, 1), (-1, 0), (0, -1))
    for level in range(levels):
        agent[level, 0] = torch.tensor([level % height, (level // height) % width])
        boxes[level, 0, 0] = torch.tensor([(level + 2) % height, (level * 3 + 1) % width])
        targets[level] = torch.tensor([(level * 2 + 1) % height, (level + 3) % width])
        for time in range(1, steps):
            agent_delta = agent_directions[(level + time) % len(agent_directions)]
            agent[level, time] = bounded_step(agent[level, time - 1], agent_delta, height, width)
            boxes[level, time] = boxes[level, time - 1]
            if time % 2 == 0:
                box_delta = box_directions[(level + time // 2) % len(box_directions)]
                boxes[level, time, 0] = bounded_step(boxes[level, time - 1, 0], box_delta, height, width)
    approach, push = future_direction_labels(agent, boxes, height, width)
    observation = torch.zeros(levels, steps, 7, height, width)
    for level in range(levels):
        for time in range(steps):
            observation[level, time, 0] = 1
            observation[level, time, 1, agent[level, time, 0], agent[level, time, 1]] = 1
            observation[level, time, 2, boxes[level, time, 0, 0], boxes[level, time, 0, 1]] = 1
            observation[level, time, 3, targets[level, 0], targets[level, 1]] = 1
    generator = torch.Generator().manual_seed(SEED)
    state = 0.02 * torch.randn(levels, steps, 12, height, width, generator=generator)
    state[:, :, :5] += torch.nn.functional.one_hot(approach, 5).permute(0, 1, 4, 2, 3)
    state[:, :, 5:10] += torch.nn.functional.one_hot(push, 5).permute(0, 1, 4, 2, 3)
    return {
        "level_id": torch.arange(levels),
        "observation": observation,
        "initial_state": state,
        "agent_xy": agent,
        "box_xy": boxes,
        "approach_label": approach,
        "push_label": push,
        "environment": "synthetic-trajectory-smoke-v1",
    }


bundle = smoke_trajectory_bundle()
train_levels = torch.arange(0, 22)
evaluation_levels = torch.arange(22, 30)
assert not set(train_levels.tolist()) & set(evaluation_levels.tolist())
{
    "mode": MODE,
    "environment": bundle["environment"],
    "train_levels": train_levels.tolist(),
    "evaluation_levels": evaluation_levels.tolist(),
    "scientific_claim_ready": False,
}

## Record all nine exact recurrent occurrences

The fixture reuses each of three cell modules once per tick. `InternalComputationSemantics` maps raw per-module call indices to `(layer, tick)` before execution. The trace must contain each coordinate exactly once per root call; selecting `(2, 2)` resolves to one declared raw occurrence rather than whichever hook output happened to run last.


In [ ]:
class TinyRepeatedDRC(torch.nn.Module):
    def __init__(self, channels=12, action_order=DIRECTIONS):
        super().__init__()
        if tuple(action_order) != DIRECTIONS:
            raise ValueError(f"action order must be {DIRECTIONS!r}")
        self.action_order = tuple(action_order)
        self.encoder = torch.nn.Conv2d(7, channels, 1, bias=False)
        self.cells = torch.nn.ModuleList(
            [
                torch.nn.Sequential(torch.nn.Conv2d(channels, channels, 1, bias=False), torch.nn.Tanh())
                for _ in range(3)
            ]
        )
        self.head = torch.nn.Linear(channels, len(self.action_order), bias=False)
        torch.nn.init.zeros_(self.encoder.weight)
        for cell in self.cells:
            torch.nn.init.dirac_(cell[0].weight)
            cell[0].weight.data.mul_(0.95)

    def forward(self, observation, state, is_init):
        reset = is_init.reshape(is_init.shape[0], *([1] * (state.ndim - 1)))
        hidden = torch.where(reset, torch.zeros_like(state), state) + self.encoder(observation)
        for _tick in range(3):
            for cell in self.cells:
                hidden = cell(hidden)
        logits = self.head(hidden.mean(dim=(-2, -1)))
        return hidden, logits


levels, steps, channels, height, width = bundle["initial_state"].shape
flat_observation = bundle["observation"].flatten(0, 1)
flat_state = bundle["initial_state"].flatten(0, 1)
batch = TensorDict(
    {
        "observation": flat_observation,
        "state": flat_state,
        "is_init": torch.zeros(levels * steps, 1, dtype=torch.bool),
    },
    batch_size=[levels * steps],
    names=["transition"],
)
policy = TensorDictModule(
    TinyRepeatedDRC(channels),
    in_keys=["observation", "state", "is_init"],
    out_keys=[("next", "state"), "logits"],
)
interaction = Interaction(policy, RecurrentSemantics.from_torchrl("state"))

cache_methods = []
for tick in range(3):
    for layer in range(3):
        path = f"module.cells.{layer}"
        target = Target(path, "activation", 1, tuple(range(channels)), occurrence=tick)
        cache_methods.append(ActivationCaching(target, cache_key=("internal", f"layer-{layer}", f"tick-{tick}")))
trace_result = run_workflow(interaction, Workflow(*cache_methods), batch.clone())

states_by_coordinates = {}
for tick in range(3):
    for layer in range(3):
        coordinates = (("layer", layer), ("tick", tick))
        path = f"module.cells.{layer}"
        value = trace_result.data["internal", f"layer-{layer}", f"tick-{tick}", path]
        states_by_coordinates[coordinates] = value.detach().reshape(levels, steps, channels, height, width)
selected_state = states_by_coordinates[SELECTED_COORDINATES]
expected_coordinates = [(("layer", layer), ("tick", tick)) for tick in range(3) for layer in range(3)]
assert list(states_by_coordinates) == expected_coordinates
{
    "recorded_coordinates": list(states_by_coordinates),
    "selected_occurrence": ("module.cells.2", 2),
    "selected_shape": tuple(selected_state.shape),
}

## Fit spatial 1x1 probes and declared controls

Each square is one example for a channel-wise linear classifier, equivalent to a 1x1 convolution with shared weights. Smoke uses three initialization seeds to stay bounded; the paper reports five. The level split remains fixed for every arm. The shuffled-label permutation is confined to the training levels, and every metric is evaluated against untouched true labels from the held-out levels.


In [ ]:
def flatten_examples(features, labels, level_indices):
    selected_features = features[level_indices].permute(0, 1, 3, 4, 2).reshape(-1, features.shape[2])
    selected_labels = labels[level_indices].reshape(-1)
    return selected_features, selected_labels


def macro_f1(prediction, target, classes=5):
    values = []
    for label in range(classes):
        true_positive = ((prediction == label) & (target == label)).sum().float()
        false_positive = ((prediction == label) & (target != label)).sum().float()
        false_negative = ((prediction != label) & (target == label)).sum().float()
        denominator = 2 * true_positive + false_positive + false_negative
        values.append(torch.where(denominator > 0, 2 * true_positive / denominator, torch.zeros_like(denominator)))
    return float(torch.stack(values).mean())


def fit_probe(features, labels, seed, *, shuffled=False):
    train_x, train_y = flatten_examples(features, labels, train_levels)
    if shuffled:
        train_y = train_y[torch.randperm(len(train_y), generator=torch.Generator().manual_seed(seed + 10_000))]
    with torch.random.fork_rng():
        torch.manual_seed(seed)
        probe = torch.nn.Linear(train_x.shape[1], len(CLASSES))
    torch.nn.init.zeros_(probe.bias)
    optimizer = torch.optim.AdamW(probe.parameters(), lr=0.01, weight_decay=0.001)
    generator = torch.Generator().manual_seed(seed)
    counts = torch.bincount(train_y, minlength=len(CLASSES)).float().clamp_min(1)
    class_weight = counts.sum() / (len(CLASSES) * counts)
    for _epoch in range(10):
        for indices in torch.randperm(len(train_x), generator=generator).split(256):
            loss = torch.nn.functional.cross_entropy(probe(train_x[indices]), train_y[indices], weight=class_weight)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
    return probe.eval()


def level_bootstrap_interval(predictions, target, draws=200):
    predictions = tuple(predictions)
    if not predictions:
        raise ValueError("bootstrap requires at least one prediction tensor")
    generator = torch.Generator().manual_seed(SEED + 99)
    values = []
    for _ in range(draws):
        sampled = torch.randint(len(evaluation_levels), (len(evaluation_levels),), generator=generator)
        values.append(
            sum(macro_f1(prediction[sampled].reshape(-1), target[sampled].reshape(-1)) for prediction in predictions)
            / len(predictions)
        )
    quantiles = torch.tensor(values).quantile(torch.tensor([0.025, 0.975]))
    return [float(value) for value in quantiles]


def evaluate_probe_family(features, labels, *, shuffled=False):
    probes = [fit_probe(features, labels, SEED + seed, shuffled=shuffled) for seed in range(3)]
    eval_x, eval_y = flatten_examples(features, labels, evaluation_levels)
    predictions = [probe(eval_x).argmax(-1).reshape(len(evaluation_levels), steps, height, width) for probe in probes]
    target = labels[evaluation_levels]
    seed_scores = [macro_f1(prediction.reshape(-1), target.reshape(-1)) for prediction in predictions]
    return probes, {
        "macro_f1_mean": float(torch.tensor(seed_scores).mean()),
        "macro_f1_seed_std": float(torch.tensor(seed_scores).std()),
        "level_bootstrap_95": level_bootstrap_interval(predictions, target),
        "seed_scores": seed_scores,
    }


probe_results = {}
trained_probes = {}
for concept, labels in (("agent_approach", bundle["approach_label"]), ("box_push", bundle["push_label"])):
    trained_probes[concept], probe_results[f"{concept}:state"] = evaluate_probe_family(selected_state, labels)
    _, probe_results[f"{concept}:observation"] = evaluate_probe_family(bundle["observation"], labels)
    shuffled, probe_results[f"{concept}:shuffled_label"] = evaluate_probe_family(selected_state, labels, shuffled=True)
    trained_probes[f"{concept}:shuffled"] = shuffled
    _, train_y = flatten_examples(selected_state, labels, train_levels)
    _, eval_y = flatten_examples(selected_state, labels, evaluation_levels)
    majority = int(torch.bincount(train_y, minlength=5).argmax())
    majority_prediction = torch.full_like(labels[evaluation_levels], majority)
    probe_results[f"{concept}:class_frequency"] = {
        "majority_class": CLASSES[majority],
        "macro_f1": macro_f1(torch.full_like(eval_y, majority), eval_y),
        "level_bootstrap_95": level_bootstrap_interval((majority_prediction,), labels[evaluation_levels]),
    }

probe_results

## Trace decoded concepts across exact internal ticks

This diagnostic holds the final-occurrence probe fixed and changes only the traced occurrence. It is descriptive formation evidence. A rising curve would not by itself show planning, search, or behavioral use.


In [ ]:
def occurrence_score(probe, features, labels):
    eval_x, eval_y = flatten_examples(features, labels, evaluation_levels)
    return macro_f1(probe(eval_x).argmax(-1), eval_y)


temporal_trace = []
for tick in range(3):
    for layer in range(3):
        coordinates = (("layer", layer), ("tick", tick))
        state = states_by_coordinates[coordinates]
        temporal_trace.append(
            {
                "layer": layer,
                "tick": tick,
                "raw_call_index": tick,
                "agent_approach_macro_f1": occurrence_score(
                    trained_probes["agent_approach"][0], state, bundle["approach_label"]
                ),
                "box_push_macro_f1": occurrence_score(trained_probes["box_push"][0], state, bundle["push_label"]),
            }
        )

temporal_trace

## Run the predeclared probe-vector intervention and matched controls

TDHook selects the declared occurrence of each reused recurrent cell directly. The selected state is then passed to a smoke-only continuation head; this is **not** the paper's recurrent intervention.

The notebook runs an explicit no-op baseline for every experimental vector. The experimental vector comes from the true-label probe; the shuffled-label vector comes from its negative control. The final random vector has the same norm and the same signed projection onto the fixed `RIGHT`-versus-`LEFT` action direction as the trained vector, while differing in its orthogonal component.


In [ ]:
class SelectedStateHead(torch.nn.Module):
    def __init__(self, source_head):
        super().__init__()
        self.steer = torch.nn.Identity()
        self.readout = torch.nn.Linear(channels, len(ACTION_INDEX), bias=False)
        self.readout.load_state_dict(source_head.state_dict())

    def forward(self, state):
        edited_state = self.steer(state)
        return edited_state, self.readout(edited_state.mean(dim=(-2, -1)))


head_module = SelectedStateHead(policy.module.head).eval()
head_policy = TensorDictModule(head_module, in_keys=["state"], out_keys=["edited_state", "logits"])
eval_state = selected_state[evaluation_levels].flatten(0, 1)
head_batch = TensorDict({"state": eval_state}, batch_size=[len(eval_state)], names=["transition"])
head_interaction = Interaction(head_policy)
box_probe = trained_probes["box_push"][0]
shuffled_box_probe = trained_probes["box_push:shuffled"][0]
target_class = CLASSES.index("RIGHT")
never_class = CLASSES.index("NEVER")
target_action = ACTION_INDEX["RIGHT"]
opposite_action = ACTION_INDEX["LEFT"]
trained_vector = (box_probe.weight[target_class] - box_probe.weight[never_class]).detach()
shuffled_vector = (shuffled_box_probe.weight[target_class] - shuffled_box_probe.weight[never_class]).detach()
action_direction = (head_module.readout.weight[target_action] - head_module.readout.weight[opposite_action]).detach()
action_unit = action_direction / action_direction.norm().clamp_min(1e-12)
parallel = torch.dot(trained_vector, action_unit)
random_direction = torch.randn(channels, generator=torch.Generator().manual_seed(SEED + 404))
orthogonal = random_direction - torch.dot(random_direction, action_unit) * action_unit
orthogonal = orthogonal / orthogonal.norm().clamp_min(1e-12)
orthogonal_norm = (trained_vector.square().sum() - parallel.square()).clamp_min(0).sqrt()
matched_vector = parallel * action_unit + orthogonal_norm * orthogonal
assert torch.allclose(matched_vector.norm(), trained_vector.norm(), atol=1e-6)
assert torch.allclose(torch.dot(matched_vector, action_unit), parallel, atol=1e-6)

TARGET_SQUARE = (1, 1)
INTERVENTION_SCALE = 4.0


def keep_state(*, output, **_):
    return output


def vector_callback(vector):
    def callback(*, output, **_):
        edited = output.clone()
        edited[:, :, TARGET_SQUARE[0], TARGET_SQUARE[1]] += INTERVENTION_SCALE * vector.to(output)
        return edited

    return callback


VECTOR_SPECS = {
    "trained_probe": trained_vector,
    "shuffled_label": shuffled_vector,
    "norm_direction_matched": matched_vector,
}


def run_vector_pair(vector):
    changed = vector_callback(vector)
    torch.manual_seed(SEED)
    baseline = run_workflow(
        head_interaction,
        Workflow(SteeringVectors(["module.steer"], steer_fn=keep_state)),
        head_batch.clone(),
    )
    torch.manual_seed(SEED)
    intervention = run_workflow(
        head_interaction,
        Workflow(SteeringVectors(["module.steer"], steer_fn=changed)),
        head_batch.clone(),
    )
    return {"baseline": baseline, "intervention": intervention}


pairs = {label: run_vector_pair(vector) for label, vector in VECTOR_SPECS.items()}
assert not any(child._forward_hooks for child in head_policy.modules())

In [ ]:
def intervention_effect(pair):
    baseline_state = pair["baseline"].data["edited_state"]
    changed_state = pair["intervention"].data["edited_state"]
    baseline_logits = pair["baseline"].data["logits"]
    changed_logits = pair["intervention"].data["logits"]
    baseline_plan = box_probe(baseline_state[:, :, TARGET_SQUARE[0], TARGET_SQUARE[1]]).argmax(-1)
    changed_plan = box_probe(changed_state[:, :, TARGET_SQUARE[0], TARGET_SQUARE[1]]).argmax(-1)
    baseline_action = baseline_logits.argmax(-1)
    changed_action = changed_logits.argmax(-1)
    return {
        "decoded_target_rate_baseline": float((baseline_plan == target_class).float().mean()),
        "decoded_target_rate_intervention": float((changed_plan == target_class).float().mean()),
        "decoded_plan_change_rate": float((changed_plan != baseline_plan).float().mean()),
        "action_target_rate_baseline": float((baseline_action == target_action).float().mean()),
        "action_target_rate_intervention": float((changed_action == target_action).float().mean()),
        "action_change_rate": float((changed_action != baseline_action).float().mean()),
    }


intervention_results = {label: intervention_effect(pair) for label, pair in pairs.items()}
control_matching = {
    "trained_norm": float(trained_vector.norm()),
    "matched_norm": float(matched_vector.norm()),
    "trained_action_projection": float(torch.dot(trained_vector, action_unit)),
    "matched_action_projection": float(torch.dot(matched_vector, action_unit)),
}
{
    "selected_occurrence": dict(SELECTED_COORDINATES),
    "target_square": TARGET_SQUARE,
    "target_concept_class": CLASSES[target_class],
    "action_order": DIRECTIONS,
    "target_action_index": target_action,
    "intervention_results": intervention_results,
    "control_matching": control_matching,
}

## Results and scope

The final object keeps execution, asset availability, statistical results, intervention mechanics, and claim readiness distinct. A positive smoke result cannot upgrade an unavailable paper checkpoint into a reproduction. A negative result is retained rather than filtered out. When paper-exact assets become available, this notebook must pin their revisions and checksums, export exact occurrence evidence from the real DRC, use the authors' frozen level split, and compare against reference metrics before the scientific fields can change.


In [ ]:
results_summary = {
    "smoke_execution": "passed",
    "paper_exact_assets": "unavailable",
    "label_generator": "notebook-local and hand-checked; reference agreement unavailable",
    "held_out_probe_results": probe_results,
    "temporal_formation": {
        "status": "smoke-only descriptive trace",
        "exact_occurrences": temporal_trace,
    },
    "intervention": {
        "status": "smoke-only selected-state continuation",
        "effects": intervention_results,
        "model_passes": {label: pair["intervention"].plan.model_passes for label, pair in pairs.items()},
    },
    "reference_agreement": "not evaluated",
    "behavioral_intervention_effect": "not evaluated on Sokoban",
    "planning_interpretation": "blocked",
    "claim_limit": "No inference to the paper checkpoint, other checkpoints, other tasks, or model-free RL broadly.",
}
results_summary